# Fixer les acteurs qui ont un siret de 9 caractères

En général, les acteurs avec un siret de 9 caractères 

## 1. Initialiser les variables d'environnement si besoin

In [ ]:
import os

# Surdéfinition des variables d'environnement permettant d'accéder à la base de données
# car je n'ai pas réussi à préciser quel fichier d'env spécifique decouple doit
# interpréter
DATABASE_URL=""
DB_WAREHOUSE=""
# Si LOG_DF = True, On affiche des contenus de dataframe dans les logs
# Note : il fauttt éviter de commiter des résultats avec des LOG_DF=True
LOG_DF=False

if DATABASE_URL:
    os.environ["DATABASE_URL"] = DATABASE_URL
if DB_WAREHOUSE:
    os.environ["DB_WAREHOUSE"] = DB_WAREHOUSE


## 2. Initialiser l'environnement

- Charger les variables d'environnment à partir d'un fichier spécifique `webapp/.env` ou `webapp/.env.preprod`
- Chargement de Django et ajout des path dans sys.path


In [ ]:
import sys
import os
from pathlib import Path

# Notebook : data-platform/notebooks
notebooks_dir = Path.cwd()
project_root = notebooks_dir.parent        # => data-platform

# On part du dossier du notebook: data-platform/notebooks
project_root = Path.cwd().parent   # => data-platform
sys.path.append(str(project_root))
sys.path.append(str(project_root / "dags"))

from dags.utils.django import django_setup_full

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

django_setup_full()


## 3. Déclaration des fonctions

- get_revision_acteur_from_vueacteur

In [ ]:
import csv

from qfdmo.models.acteur import RevisionActeur, Acteur

EMPTY = '__empty__'
DRY_RUN = False

def get_revision_acteur_from_vueacteur(vueacteur):
    if vueacteur.est_parent:
        return RevisionActeur.objects.get(identifiant_unique=vueacteur.identifiant_unique)
    else:
        acteur = Acteur.objects.get(identifiant_unique=vueacteur.identifiant_unique)
        return acteur.get_or_create_revision()


## 4. récupération des acteurs visible (carte ou open-data)

In [ ]:
from django.db.models import Q
from django.db.models.functions import Length

from qfdmo.models.acteur import VueActeur

acteurs = VueActeur.objects.filter(Q(est_dans_opendata=True) | Q(est_dans_carte=True))

print(acteurs.count())


## 5. Récupération des acteurs

- dont le SIRET est une chaine de caractère de 9 chiffres
- dont le SIRET est une chaine de caractère de 9 chiffres et le siren est vide
- dont le SIRET est une chaine de caractère de 9 chiffres et le siren = siret

In [ ]:
from django.db.models import F

acteurs_siret_9_digits = acteurs.filter(siret__regex=r'^[0-9]{9}$')

acteurs_siret_9_no_siren = acteurs_siret_9_digits.filter(siren='')

print(
    " dont le SIRET est une chaine de caractère de 9 chiffres et le siren est vide: "
    f"{acteurs_siret_9_no_siren.count()}"
)

acteurs_siret_equals_siren = acteurs_siret_9_digits.filter(siren=F('siret'))

print(
    "dont le SIRET est une chaine de caractère de 9 chiffres et le siren = siret: "
    f"{acteurs_siret_equals_siren.count()}"
)



In [ ]:

result = [['identifiant_unique', 'nom', 'siren origin', 'siren cible', 'siret origin', 'siret cible']]
for vueacteur in acteurs_siret_9_no_siren:
    revision_acteur = get_revision_acteur_from_vueacteur(vueacteur)

    print(f"Acteur {vueacteur.identifiant_unique} - {vueacteur.nom}")
    print(f"  SIREN from `{revision_acteur.siren}` to `{vueacteur.siret}`")
    print(f"  SIRET from `{revision_acteur.siret}` to `{EMPTY}`")
    target_siren = vueacteur.siret
    target_siret = EMPTY
    result.append([vueacteur.identifiant_unique, vueacteur.nom, vueacteur.siren, target_siren, vueacteur.siret, target_siret])
    if not DRY_RUN:
        revision_acteur.siren = target_siren
        revision_acteur.siret = target_siret
        revision_acteur.save()
csv.writer(open('result_acteurs_siret_9_no_siren.csv', 'w')).writerows(result)


In [ ]:

result = [['identifiant_unique', 'nom', 'siren', 'siret origin', 'siret cible']]
for vueacteur in acteurs_siret_equals_siren:
    revision_acteur = get_revision_acteur_from_vueacteur(vueacteur)
    
    print(f"Acteur {vueacteur.identifiant_unique} - {vueacteur.nom}")
    print(f"  SIRET from `{revision_acteur.siret}` to `{EMPTY}`")
    target_siret = EMPTY
    result.append([vueacteur.identifiant_unique, vueacteur.nom, vueacteur.siren, vueacteur.siret, target_siret])
    if not DRY_RUN:
        revision_acteur.siret = target_siret
        revision_acteur.save()
csv.writer(open('result_acteurs_siret_equals_siren.csv', 'w')).writerows(result)
